# Gemma-2 9B QLoRA inference

This notebook loads the trained adapter, performs 2,048-token inference, and
writes a valid three-class probability submission. Paths are configured through
environment variables so the notebook can run on Kaggle or another GPU host.

In [ ]:
!pip install -q transformers==4.42.3 bitsandbytes==0.43.1 accelerate==0.32.1 peft==0.11.1

In [ ]:
import ast
import os
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import BitsAndBytesConfig, Gemma2ForSequenceClassification, GemmaTokenizerFast
from peft import PeftModel

@dataclass
class Config:
    data_dir: str = os.environ.get("LLM_DATA_DIR", "/kaggle/input/llm-classification-finetuning")
    base_model: str = os.environ.get("LLM_BASE_MODEL", "unsloth/gemma-2-9b-it-bnb-4bit")
    adapter_dir: str = os.environ.get("LLM_ADAPTER_DIR", "gemma2_qlora_output")
    output_path: str = os.environ.get("LLM_SUBMISSION_PATH", "submission.csv")
    max_length: int = 2048
    batch_size: int = 8
    use_tta: bool = os.environ.get("LLM_USE_TTA", "0") == "1"

cfg = Config()
print(cfg)

In [ ]:
def parse_text(value):
    if isinstance(value, list):
        return " ".join(str(item) for item in value if item is not None)
    if not isinstance(value, str):
        return str(value)
    try:
        parsed = ast.literal_eval(value)
    except (SyntaxError, ValueError):
        return value
    return " ".join(str(item) for item in parsed if item is not None) if isinstance(parsed, list) else str(parsed)

test = pd.read_csv(Path(cfg.data_dir) / "test.csv")
for column in ["prompt", "response_a", "response_b"]:
    test[column] = test[column].map(parse_text)
print(f"test rows: {len(test):,}")

In [ ]:
tokenizer = GemmaTokenizerFast.from_pretrained(cfg.base_model)
tokenizer.add_eos_token = True
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

def make_texts(data, swapped=False):
    a = data["response_b"] if swapped else data["response_a"]
    b = data["response_a"] if swapped else data["response_b"]
    return [
        "<prompt>: " + p + "\n\n<response_a>: " + x + "\n\n<response_b>: " + y
        for p, x, y in zip(data["prompt"], a, b)
    ]

def predict(data, model):
    probabilities = []
    model.eval()
    for start in range(0, len(data), cfg.batch_size):
        batch = tokenizer(
            make_texts(data.iloc[start:start + cfg.batch_size]),
            max_length=cfg.max_length,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )
        batch = {key: value.to(model.device) for key, value in batch.items()}
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            probabilities.append(model(**batch).logits.softmax(-1).float().cpu().numpy())
    return np.concatenate(probabilities)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base = Gemma2ForSequenceClassification.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    num_labels=3,
)
base.config.pad_token_id = tokenizer.pad_token_id
model = PeftModel.from_pretrained(base, cfg.adapter_dir)
original = predict(test, model)

if cfg.use_tta:
    swapped = predict(test.assign(response_a=test.response_b, response_b=test.response_a), model)
    swapped = swapped[:, [1, 0, 2]]
    probabilities = 0.45 * original + 0.55 * swapped
else:
    probabilities = original

probabilities = np.clip(probabilities, 0.0, None)
probabilities /= probabilities.sum(axis=1, keepdims=True)
submission = pd.DataFrame({
    "id": test["id"],
    "winner_model_a": probabilities[:, 0],
    "winner_model_b": probabilities[:, 1],
    "winner_tie": probabilities[:, 2],
})
submission.to_csv(cfg.output_path, index=False)
print(f"wrote {len(submission):,} rows to {cfg.output_path}")
display(submission)

## Output contract

The generated `submission.csv` contains one row per test ID and three
non-negative probabilities whose rows sum to one. The default inference length
is 2,048 tokens; response-order TTA can be enabled with
`LLM_USE_TTA=1`.